Prévoir 4-5 heures d'entrainement.

## Entraînement de l'Ensemble avec Bagging (Stochasticité des données)

Ce notebook sert uniquement à entraîner 3 modèles sur des sous-échantillons bootstrapés et à sauvegarder les poids.

Imports et préparation des données

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from sklearn.model_selection import train_test_split

# 1. Chargement des données
p_attr = pd.read_json("data/Face4Shifts/Anno/p_attr.json", lines=True)

# 2. Création de la variable proxy
p_attr["label"] = (p_attr["long_hair"] == 1) & ((p_attr["smile_with_closed_lips"] == 1) | (p_attr["smile_with_open_lips"] == 1))

# 3. Séparation des données
# On garde le random_state=42 pour que le jeu d'entraînement global soit identique
X = p_attr.drop(columns=["label"])
y = p_attr["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Taille du jeu d'entraînement global : {len(X_train)} images")

Taille du jeu d'entraînement global : 24000 images


Définition du Dataset

In [2]:
class FaceDataset(Dataset):
    def __init__(self, img_dir, data, label, transform=None):
        self.img_dir = img_dir
        self.data = data
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.iloc[idx]["ID"]
        label = self.label.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{img_id.strip()}.jpg")
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# Transformations standard
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Dataset global d'entraînement
train_dataset = FaceDataset(img_dir="data/Face4Shifts/Img/Photo", data=X_train, label=y_train, transform=transform)

Boucle d'entraînement avec Bootstrap Aggregating (Bagging)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Entraînement sur : {device}")

NUM_MODELS = 3
NUM_EPOCHS = 5

for k in range(1, NUM_MODELS + 1):
    print(f"\n{'='*50}")
    print(f" DÉMARRAGE DE L'ENTRAÎNEMENT DU MODÈLE {k}/{NUM_MODELS} (BAGGING)")
    print(f"{'='*50}")
    
    # --- LA MAGIE DU BAGGING EST ICI ---
    # 1. On tire au hasard des indices avec remise (replace=True)
    bootstrap_indices = np.random.choice(len(train_dataset), size=len(train_dataset), replace=True)
    
    # 2. On crée le sous-dataset bootstrapé
    bagging_dataset = Subset(train_dataset, bootstrap_indices)
    
    # 3. On crée un DataLoader SPÉCIFIQUE pour ce modèle
    bagging_dataloader = DataLoader(bagging_dataset, batch_size=32, shuffle=True)
    # -----------------------------------
    
    # Instanciation d'un modèle neuf
    model = models.efficientnet_b0(weights="EfficientNet_B0_Weights.DEFAULT")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    model = model.to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # Boucle d'entraînement
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        
        # On itère sur le bagging_dataloader !
        for i, (images, labels) in enumerate(bagging_dataloader):
            images, labels = images.to(device), labels.float().to(device)
            
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
            if i % 100 == 0 and i > 0:
                print(f"   Modèle {k} | Epoch {epoch+1}/{NUM_EPOCHS} | Batch {i} | Loss: {loss.item():.4f}")
                
        # Calcul de la loss sur la taille de l'échantillon bootstrapé
        epoch_loss = running_loss / len(bagging_dataset)
        print(f" -> Modèle {k} | FIN Epoch {epoch+1} | Loss moyenne: {epoch_loss:.4f}")
        
    # Sauvegarde avec un nom distinctif
    model_path = f"./model_ens_bagging_{k}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"Modèle {k} sauvegardé sous : {model_path}")

print("\nEntraînement de l'Ensemble avec Bagging terminé !")

Entraînement sur : cpu

 DÉMARRAGE DE L'ENTRAÎNEMENT DU MODÈLE 1/3 (BAGGING)
   Modèle 1 | Epoch 1/5 | Batch 100 | Loss: 0.2275
   Modèle 1 | Epoch 1/5 | Batch 200 | Loss: 0.3665
   Modèle 1 | Epoch 1/5 | Batch 300 | Loss: 0.2204
   Modèle 1 | Epoch 1/5 | Batch 400 | Loss: 0.1317
   Modèle 1 | Epoch 1/5 | Batch 500 | Loss: 0.0888
   Modèle 1 | Epoch 1/5 | Batch 600 | Loss: 0.1734
   Modèle 1 | Epoch 1/5 | Batch 700 | Loss: 0.2027
 -> Modèle 1 | FIN Epoch 1 | Loss moyenne: 0.2480
   Modèle 1 | Epoch 2/5 | Batch 100 | Loss: 0.1222
   Modèle 1 | Epoch 2/5 | Batch 200 | Loss: 0.0563
   Modèle 1 | Epoch 2/5 | Batch 300 | Loss: 0.0636
   Modèle 1 | Epoch 2/5 | Batch 400 | Loss: 0.0860
   Modèle 1 | Epoch 2/5 | Batch 500 | Loss: 0.0564
   Modèle 1 | Epoch 2/5 | Batch 600 | Loss: 0.0431
   Modèle 1 | Epoch 2/5 | Batch 700 | Loss: 0.0417
 -> Modèle 1 | FIN Epoch 2 | Loss moyenne: 0.1230
   Modèle 1 | Epoch 3/5 | Batch 100 | Loss: 0.0336
   Modèle 1 | Epoch 3/5 | Batch 200 | Loss: 0.0445
   Modè